# 05 — Sentinel-1 : traitement RTC, composites saisonniers et couches (tâche D5)

Chaîne complète pour la piste 132 (ascendante) de Sentinel-1 :

1. choisir les dates dans `inventaire_s1_douala.csv` (saison sèche et saison des pluies) ;
2. faire calibrer les scènes par **HyP3** (service gratuit d'ASF, produit RTC γ⁰ en puissance, 30 m) : on ne télécharge donc pas les scènes brutes de 1 Go ;
3. recadrer sur la bbox du contrat (1400 × 1200 px, EPSG:4326) ;
4. produire, par date puis par saison : rétrodiffusion VV, masque d'eau, fréquence d'eau, « nouvelle eau » ;
5. mesurer l'écart éventuel entre Sentinel-1A et Sentinel-1D.

⚠️ Sentinel-1 est en **bande C** (VV/VH), le contrat v1.0 est écrit pour NISAR (HH/HV, bande L).
Les noms `backscatter_vv` et les composites sortent du contrat : il faudra une **v2.0 validée par l'équipe**.

⚠️ Le jeton Earthdata se place dans les **Secrets Colab** sous le nom `EARTHDATA_TOKEN`. Ne jamais l'écrire dans ce notebook.

In [ ]:
!pip -q install hyp3_sdk rasterio pillow pandas numpy matplotlib

## 1. Paramètres

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

CSV        = "inventaire_s1_douala.csv"     # produit par le notebook D6 (à importer dans Colab)
TRACK      = 132                            # piste ascendante qui couvre 100 % de la bbox
BBOX       = (9.55, 3.90, 9.90, 4.20)       # ouest, sud, est, nord (contrat v1.0)
WIDTH, HEIGHT = 1400, 1200                  # taille des PNG (contrat)

DRY_MONTHS  = (12, 1, 2)                    # saison sèche (contrat)
RAIN_MONTHS = (7, 8, 9, 10)                 # coeur de la saison des pluies à Douala
MAX_PER_SEASON = 8                          # nombre maximal de dates par saison
CHECK_DATES = ["2026-06-25", "2026-07-02"]  # dernier passage S1A / premier passage S1D (test d'écart)

THRESH_VV_DB = -14.0    # mesuré sur les 16 dates réelles : pic de l'eau vers -17 dB, creux vers -13 dB (à valider avec l'équipe)
FREQ_WATER   = 0.5      # un pixel est « en eau » pour une saison s'il l'est sur au moins 50 % des dates
SUBMIT       = False    # passer à True seulement après avoir lu le récapitulatif de la section 3
JOB_NAME     = "douala-rtc"

WORK   = Path("work")
RTC_DIR  = WORK / "rtc"        # produits HyP3 (VV/VH)
GRID_DIR = WORK / "grilles"    # une grille 1400x1200 par date (petits fichiers .npy)
OUT_DIR  = Path("sortie_s1") / "layers"
for d in (RTC_DIR, GRID_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Choix des dates

In [ ]:
inv = pd.read_csv(CSV)
inv = inv[inv["piste"] == TRACK].drop_duplicates("date").sort_values("date").reset_index(drop=True)
inv["mois"] = inv["date"].str[5:7].astype(int)

def etale(df, n):
    if len(df) <= n:
        return df
    idx = np.unique(np.linspace(0, len(df) - 1, n).round().astype(int))
    return df.iloc[idx]

seche  = etale(inv[inv["mois"].isin(DRY_MONTHS)  & (inv["date"] >= "2025-12-01") & (inv["date"] <= "2026-02-28")], MAX_PER_SEASON).assign(saison="seche")
pluies = etale(inv[inv["mois"].isin(RAIN_MONTHS) & (inv["date"] >= "2026-07-01")], MAX_PER_SEASON).assign(saison="pluies")
check  = inv[inv["date"].isin(CHECK_DATES)].assign(saison="controle")

dates = pd.concat([seche, pluies, check]).drop_duplicates("date").sort_values("date").reset_index(drop=True)
print(dates[["date", "saison", "satellite", "couv_bbox_%", "taille_MB", "scene"]].to_string())
print()
print(len(dates), "scènes à traiter ; sèche :", len(seche), "| pluies :", len(pluies))
print("Satellites :", dates.groupby(["saison", "satellite"]).size().to_dict())

## 3. Connexion à HyP3 et récapitulatif

HyP3 est gratuit mais limité par des **crédits mensuels**. Le récapitulatif ci-dessous affiche les crédits restants ;
vérifiez-les avant de passer `SUBMIT = True`.

In [ ]:
from google.colab import userdata
import hyp3_sdk as sdk

token = userdata.get("EARTHDATA_TOKEN")      # Secrets Colab (icône clé, à gauche)
hyp3 = sdk.HyP3(token=token)
print("Crédits HyP3 restants :", hyp3.check_credits())

deja = hyp3.find_jobs(name=JOB_NAME)
print(len(deja), "jobs déjà soumis sous le nom", JOB_NAME)

In [ ]:
deja_granules = set()
for j in deja:
    if j.failed():
        continue
    for g in (j.job_parameters or {}).get("granules", []):
        deja_granules.add(g)

a_soumettre = [s for s in dates["scene"] if s not in deja_granules]
print(len(a_soumettre), "scènes à soumettre,", len(deja_granules), "déjà présentes")

if SUBMIT and a_soumettre:
    batch = sdk.Batch()
    for s in a_soumettre:
        batch += hyp3.submit_rtc_job(s, name=JOB_NAME, radiometry="gamma0", scale="power", resolution=30)
    print("Soumis :", len(batch), "| coût en crédits :", batch.total_credit_cost())
else:
    print("SUBMIT = False : rien n'a été envoyé.")

## 4. Attente et téléchargement des produits RTC

In [ ]:
import zipfile

jobs = hyp3.find_jobs(name=JOB_NAME)
print(jobs)
jobs = hyp3.watch(jobs)                         # attend la fin (peut durer plusieurs dizaines de minutes)
ok = jobs.filter_jobs(succeeded=True, running=False, pending=False, failed=False)
print(len(ok), "jobs réussis sur", len(jobs))

zips = ok.download_files(RTC_DIR)
for z in zips:
    with zipfile.ZipFile(z) as zf:
        for m in zf.namelist():
            if m.endswith("_VV.tif") or m.endswith("_VH.tif"):
                zf.extract(m, RTC_DIR)
    Path(z).unlink()                            # on libère le disque
print(sorted(p.name for p in RTC_DIR.rglob("*_VV.tif"))[:5], "...")

## 5. Recadrage sur la bbox : une grille 1400 × 1200 par date

Chaque produit HyP3 est en projection UTM. On le reprojette sur la grille du contrat (EPSG:4326, nord en haut) et on garde la **puissance linéaire**
(la moyenne se fait en linéaire, jamais en dB).

In [ ]:
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

DST_TRANSFORM = from_bounds(*BBOX, WIDTH, HEIGHT)

def date_et_sat(path):
    nom = Path(path).name                       # ex. S1D_IW_20260702T173540_DVP_RTC30_G_gpuned_AB12_VV.tif
    parts = nom.split("_")
    d = parts[2][:8]
    return f"{d[:4]}-{d[4:6]}-{d[6:]}", parts[0]

def vers_grille(path):
    dst = np.full((HEIGHT, WIDTH), np.nan, dtype="float32")
    with rasterio.open(path) as src:
        reproject(source=rasterio.band(src, 1), destination=dst,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=0,
                  dst_transform=DST_TRANSFORM, dst_crs="EPSG:4326", dst_nodata=np.nan,
                  resampling=Resampling.bilinear)
    dst[~(dst > 0)] = np.nan                    # 0 ou négatif = pas de donnée
    return dst

meta = {}
for pol in ("VV", "VH"):
    for p in sorted(RTC_DIR.rglob(f"*_{pol}.tif")):
        d, sat = date_et_sat(p)
        sortie = GRID_DIR / f"{d}_{pol.lower()}.npy"
        if not sortie.exists():
            np.save(sortie, vers_grille(p))
        meta[d] = sat
print(len(meta), "dates mises en grille :", sorted(meta)[:3], "...")

In [ ]:
def charge(d, pol="vv"):
    return np.load(GRID_DIR / f"{d}_{pol}.npy")

def en_db(p):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(p)

saison_de = dict(zip(dates["date"], dates["saison"]))
dispo = sorted(d for d in saison_de if (GRID_DIR / f"{d}_vv.npy").exists())
manquantes = sorted(set(saison_de) - set(dispo))
print(len(dispo), "dates prêtes ;", "manquantes :", manquantes)

## 6. Seuil d'eau : à valider avec l'histogramme

In [ ]:
import matplotlib.pyplot as plt

ex = [d for d in dispo if saison_de[d] == "pluies"][:1] + [d for d in dispo if saison_de[d] == "seche"][:1]
if not ex:
    raise SystemExit("Aucune grille disponible : exécutez d'abord les sections 4 et 5 (jobs HyP3 terminés, produits téléchargés et mis en grille).")
fig, axes = plt.subplots(1, len(ex), figsize=(6 * len(ex), 3.5), squeeze=False)
for ax, d in zip(axes[0], ex):
    v = en_db(charge(d))
    ax.hist(v[np.isfinite(v)], bins=200, range=(-35, 5))
    ax.axvline(THRESH_VV_DB, color="red")
    ax.set_title(f"{d} - VV (dB), seuil {THRESH_VV_DB}")
plt.show()
print("Cherchez le creux entre le pic de l'eau (valeurs basses) et celui de la terre ; ajustez THRESH_VV_DB.")

## 7. Couches par date et statistiques

Format du contrat : PNG RGBA 1400 × 1200 ; rétrodiffusion en gris sur [-30, 0] dB avec alpha 0 sans donnée ;
masque d'eau cyan (34, 211, 238) d'alpha 170.

In [ ]:
from PIL import Image

KM2_PIXEL = ((BBOX[2] - BBOX[0]) / WIDTH * 111.195 * np.cos(np.radians((BBOX[1] + BBOX[3]) / 2))) * ((BBOX[3] - BBOX[1]) / HEIGHT * 111.195)

def png_gris(db, chemin):
    v = np.clip((db + 30) / 30, 0, 1)
    g = np.nan_to_num(v * 255).astype("uint8")
    a = np.where(np.isfinite(db), 255, 0).astype("uint8")
    Image.fromarray(np.dstack([g, g, g, a]), "RGBA").save(chemin, optimize=True)

def png_masque(masque, couleur, alpha, chemin):
    img = np.zeros((HEIGHT, WIDTH, 4), dtype="uint8")
    img[masque] = (*couleur, alpha)
    Image.fromarray(img, "RGBA").save(chemin, optimize=True)

CYAN, ORANGE = (34, 211, 238), (251, 146, 60)
lignes = []
for d in dispo:
    db = en_db(charge(d))
    valide = np.isfinite(db)
    eau = valide & (db < THRESH_VV_DB)
    png_gris(db, OUT_DIR / f"{d}_backscatter_vv.png")
    png_masque(eau, CYAN, 170, OUT_DIR / f"{d}_water_mask.png")
    lignes.append({"date": d, "satellite": meta.get(d), "saison": saison_de[d],
                   "valid_fraction": round(float(valide.mean()), 3),
                   "water_area_km2": round(float(eau.sum() * KM2_PIXEL), 2),
                   "mean_vv_db": round(float(np.nanmean(db)), 2),
                   "seuil_vv_db": THRESH_VV_DB})
stats = pd.DataFrame(lignes)
stats

## 8. Composites saisonniers

- Rétrodiffusion : moyenne en **puissance linéaire** sur les dates de la saison, puis conversion en dB.
- Eau : fréquence = part des dates valides où le pixel est sous le seuil.
- « Nouvelle eau » : en eau pendant les pluies (fréquence ≥ seuil) mais pas en saison sèche.

In [ ]:
comp = {}
for saison in ("seche", "pluies"):
    ds = [d for d in dispo if saison_de[d] == saison]
    if not ds:
        print("Aucune date pour", saison)
        continue
    pile = np.stack([charge(d) for d in ds])                    # puissance linéaire
    moy = np.nanmean(pile, axis=0)
    db = en_db(moy)
    n_valid = np.isfinite(pile).sum(axis=0)
    eau = np.where(np.isfinite(pile), en_db(pile) < THRESH_VV_DB, False).sum(axis=0)
    freq = np.where(n_valid > 0, eau / np.maximum(n_valid, 1), np.nan)
    comp[saison] = {"db": db, "freq": freq, "n": len(ds)}
    png_gris(db, OUT_DIR / f"composite-{saison}_backscatter_vv.png")
    png_masque(np.nan_to_num(freq) >= FREQ_WATER, CYAN, 170, OUT_DIR / f"composite-{saison}_water_mask.png")
    print(saison, ":", len(ds), "dates | eau (freq >=", FREQ_WATER, ") :", round(float((np.nan_to_num(freq) >= FREQ_WATER).sum() * KM2_PIXEL), 1), "km2")

if len(comp) == 2:
    nouvelle = (np.nan_to_num(comp["pluies"]["freq"]) >= FREQ_WATER) & (np.nan_to_num(comp["seche"]["freq"], nan=1.0) < FREQ_WATER)
    png_masque(nouvelle, ORANGE, 190, OUT_DIR / "composite-pluies_new_water_mask.png")
    print("Nouvelle eau :", round(float(nouvelle.sum() * KM2_PIXEL), 1), "km2")

## 8 bis. Zone de démarcation eau / terre et carte de changement

L'eau libre de l'estuaire change peu entre saisons ; l'information utile est dans la **frontière** entre l'eau et la terre :

- **fréquence d'eau** : part des dates où chaque pixel est en eau (toutes dates confondues) ;
- **eau permanente** = toujours en eau ; **emprise maximale** = au moins une fois en eau ;
- **zone de démarcation** = entre les deux (pixel parfois en eau, parfois à sec) : berges, vasières, bord de mangrove ;
- **carte de changement** = rétrodiffusion des pluies moins celle de la saison sèche (dB) : bleu = plus sombre, rouge = plus clair ;
- **variabilité temporelle** = écart-type de la rétrodiffusion sur toutes les dates.

⚠️ Ces écarts mélangent plusieurs causes : **saison, marée, vent** et, entre sèche (S1A) et pluies (S1D), un éventuel **décalage entre satellites** (section 9).
La marée n'est pas contrôlée ici ; sans données de marée, on ne peut pas attribuer la démarcation à la saison seule.

In [ ]:
from matplotlib import colormaps

DEMARC_MIN, DEMARC_MAX = 0.1, 0.9      # un pixel est « en démarcation » s'il est en eau entre 10 % et 90 % des dates

pile_db = np.stack([en_db(charge(d)) for d in dispo])
valides = np.isfinite(pile_db)
eau_t = valides & (pile_db < THRESH_VV_DB)
freq_all = np.where(valides.sum(axis=0) > 0, eau_t.sum(axis=0) / np.maximum(valides.sum(axis=0), 1), np.nan)
ok_pix = np.isfinite(freq_all)

permanente = ok_pix & (freq_all >= 0.999)
maximale   = ok_pix & (freq_all > 0)
marge      = ok_pix & (freq_all > 0) & (freq_all < 0.999)
demarcation = ok_pix & (freq_all >= DEMARC_MIN) & (freq_all <= DEMARC_MAX)

def png_carte(valeurs, cmap, vmin, vmax, chemin, alpha=230):
    norm = np.clip((np.nan_to_num(valeurs, nan=vmin) - vmin) / (vmax - vmin), 0, 1)
    img = (colormaps[cmap](norm) * 255).astype("uint8")
    img[..., 3] = np.where(np.isfinite(valeurs), alpha, 0)
    Image.fromarray(img, "RGBA").save(chemin, optimize=True)

png_carte(freq_all, "viridis", 0, 1, OUT_DIR / "frequence_eau.png")
png_masque(demarcation, (250, 204, 21), 200, OUT_DIR / "zone_demarcation.png")
png_masque(permanente, CYAN, 170, OUT_DIR / "eau_permanente.png")
png_masque(maximale, CYAN, 120, OUT_DIR / "emprise_maximale.png")
sd = np.nanstd(pile_db, axis=0)
png_carte(sd, "magma", 0, 5, OUT_DIR / "variabilite_temporelle.png")

if len(comp) == 2:
    changement = comp["pluies"]["db"] - comp["seche"]["db"]
    png_carte(changement, "RdBu_r", -4, 4, OUT_DIR / "changement_pluies_moins_seche.png")
    print("Changement : médiane", round(float(np.nanmedian(changement)), 2), "dB |",
          "hausse > 2 dB :", round(float(np.nansum(changement > 2) * KM2_PIXEL), 1), "km2 |",
          "baisse < -2 dB :", round(float(np.nansum(changement < -2) * KM2_PIXEL), 1), "km2")

resume = {
    "dates": len(dispo),
    "eau_permanente_km2": round(float(permanente.sum() * KM2_PIXEL), 1),
    "emprise_maximale_km2": round(float(maximale.sum() * KM2_PIXEL), 1),
    "marge_variable_km2": round(float(marge.sum() * KM2_PIXEL), 1),
    "demarcation_10_90_km2": round(float(demarcation.sum() * KM2_PIXEL), 1),
    "seuil_vv_db": THRESH_VV_DB,
}
print(resume)
pd.Series(resume).to_csv("sortie_s1/stats_demarcation.csv", header=["valeur"])

## 9. Écart entre Sentinel-1A et Sentinel-1D

Les deux passages (25/06 en S1A, 02/07 en S1D) sont séparés de 7 jours. On compare les pixels « stables » (ni eau ni valeurs extrêmes).
Un écart médian faible (de l'ordre de quelques dixièmes de dB) peut être noté comme limite ; un écart plus fort doit être corrigé ou signalé.

In [ ]:
a, b = CHECK_DATES
if a in dispo and b in dispo:
    da, db_ = en_db(charge(a)), en_db(charge(b))
    moyenne = (da + db_) / 2
    stable = np.isfinite(da) & np.isfinite(db_) & (moyenne > -12) & (moyenne < -5)
    diff = (db_ - da)[stable]
    print(f"Pixels stables : {stable.sum()} ({100 * stable.mean():.0f} % de la bbox)")
    print(f"Écart S1D - S1A : médiane {np.median(diff):+.2f} dB | intervalle interquartile {np.percentile(diff, 25):+.2f} à {np.percentile(diff, 75):+.2f} dB")
    print("Interprétation : la médiane mesure un décalage éventuel entre satellites ; elle contient aussi les vrais changements en 7 jours.")
else:
    print("Dates de contrôle absentes :", [d for d in CHECK_DATES if d not in dispo])

## 10. Export

In [ ]:
import shutil
stats.to_csv("sortie_s1/stats_s1.csv", index=False)
archive = shutil.make_archive("sortie_s1", "zip", "sortie_s1")
print("Archive :", archive, "-", round(Path(archive).stat().st_size / 1e6, 1), "Mo")

# from google.colab import files; files.download(archive)      # téléchargement sur votre ordinateur
# Optionnel : copie sur le Drive partagé
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(archive, "/content/drive/MyDrive/<dossier_partage>/")

## Fini quand (D5)
- [ ] Les scènes sont traitées par HyP3 et les grilles `work/grilles/*.npy` existent pour toutes les dates
- [ ] Le seuil d'eau a été validé avec l'histogramme et noté dans `stats_s1.csv`
- [ ] `sortie_s1.zip` (PNG par date + composites + `stats_s1.csv`) est partagé avec l'équipe
- [ ] L'écart S1A/S1D est noté comme limite
- [ ] L'équipe a décidé de la v2.0 du contrat (VV/VH, composites, seuil)